In [9]:
import os
import numpy as np
import tensorflow as tf
from flask import Flask, render_template, request, jsonify
from werkzeug.utils import secure_filename
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Use current working directory for Jupyter environment compatibility
BASE_DIR = os.getcwd()

# Global Project Settings
MODEL_PATH = os.path.join(BASE_DIR, 'Skin_Diseases.h5')
UPLOAD_FOLDER = os.path.join(BASE_DIR, 'uploads')
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg'}
IMG_SIZE = (64, 64)
CLASSES = ['Acne', 'Melanoma', 'Psoriasis', 'Rosacea', 'Vitiligo']

# Guarantee upload directory existence
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

print(f"[STATUS]: Project Directories Initialized Successfully.")
print(f"Base Directory: {BASE_DIR}")
print(f"Uploads Directory: {UPLOAD_FOLDER}")

[STATUS]: Project Directories Initialized Successfully.
Base Directory: /Users/esheta/DL_Skin_Disease
Uploads Directory: /Users/esheta/DL_Skin_Disease/uploads


In [10]:
print("[INFO]: Compiling Transfer Learning Architecture using MobileNetV2 Base...")

# 1. Instantiate state-of-the-art Feature Extractor pre-trained on ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), 
    include_top=False, 
    weights='imagenet'
)
base_model.trainable = False  # Freeze pre-trained feature extraction weights

# 2. Append Custom Diagnostic Classification Head
inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)  # Dropout layer to minimize overfitting risks
outputs = layers.Dense(len(CLASSES), activation='softmax')(x)

# 3. Assemble and Compile Model Graph
model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 4. Save compiled model snapshot to disk
model.save(MODEL_PATH)
print(f"[SUCCESS]: New model architecture compiled and written to: {MODEL_PATH}")

[INFO]: Compiling Transfer Learning Architecture using MobileNetV2 Base...


/var/folders/_k/l7z8b1sx6_75xbzzhy2cgpz00000gn/T/ipykernel_7510/3865772777.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


[SUCCESS]: New model architecture compiled and written to: /Users/esheta/DL_Skin_Disease/Skin_Diseases.h5


In [11]:
print("[INFO]: Initializing model memory allocation sequence...")

if os.path.exists(MODEL_PATH):
    # Load model object directly into runtime memory
    loaded_model = load_model(MODEL_PATH)
    print("[SUCCESS]: Deep Learning model successfully loaded into active RAM!")
else:
    raise FileNotFoundError(f"[ERROR]: System could not locate the model at {MODEL_PATH}")

[INFO]: Initializing model memory allocation sequence...


[SUCCESS]: Deep Learning model successfully loaded into active RAM!


In [12]:
app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 5 * 1024 * 1024 # Secure constraint: 5MB upload ceiling

def allowed_file(filename):
    """Secure validation of file extensions."""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

@app.route('/', methods=['GET'])
def index():
    """Renders the main frontend user interface."""
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    """Processes images, runs inference through MobileNetV2, and streams back JSON."""
    if 'file' not in request.files:
        return jsonify({'error': 'Payload parsing failed: No file parameter found.'}), 400
        
    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'Payload processing failed: Empty file field selection.'}), 400
        
    if file and allowed_file(file.filename):
        filename = secure_filename(file.filename)
        file_path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        
        try:
            # Commit file asset to temporary server storage disk
            file.save(file_path)
            
            # Formulate evaluation matrix matching input pipeline dimensions (64, 64, 3)
            img = image.load_img(file_path, target_size=IMG_SIZE)
            x = image.img_to_array(img) / 255.0  # Min-Max Normalization scaling pixel matrix to [0,1]
            x = np.expand_dims(x, axis=0)        # Append batch size dimension block

            # Execute Core Inference Routine
            predictions = loaded_model.predict(x)
            predicted_class_idx = np.argmax(predictions, axis=-1)[0]
            confidence_score = float(predictions[0][predicted_class_idx])
            
            # Disk Cleanup: Delete image post-inference to prevent backend storage bloating
            if os.path.exists(file_path):
                os.remove(file_path)

            # Return standardized RESTful JSON output response
            return jsonify({
                'success': True,
                'prediction': CLASSES[predicted_class_idx],
                'confidence': f"{round(confidence_score * 100, 2)}%",
                'distribution_metrics': {CLASSES[i]: f"{round(float(prob) * 100, 2)}%" for i, prob in enumerate(predictions[0])}
            }), 200

        except Exception as e:
            if os.path.exists(file_path):
                os.remove(file_path)
            return jsonify({'error': f'Runtime Execution Failure: {str(e)}'}), 500
    else:
        return jsonify({'error': 'Unsupported file format extension. Allowed types: png, jpg, jpeg.'}), 400

In [ ]:
if __name__ == '__main__':
    print("[RUNNING]: Launching Multi-threaded Local Server on http://127.0.0.1:5000")
    # use_reloader=False stops Jupyter from crashing your notebook kernel loop
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

[RUNNING]: Launching Multi-threaded Local Server on http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 14:38:02] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 14:38:02] "GET /static/js/main.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 14:38:02] "GET /static/css/main.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 14:38:04] "GET /favicon.ico HTTP/1.1" 404 -
2026-06-03 14:38:19.005967: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 14:38:21] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 14:38:33] "POST /predict HTTP/1.1" 200 -
